In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table= f"{catalog_name}.{bronze_schema}.drivers"
silver_table= f"{catalog_name}.{silver_schema}.drivers"

In [0]:
from pyspark.sql import functions as F

In [0]:
drivers_df = spark.table(bronze_table)

In [0]:
drivers_drop_df= drivers_df.drop("url")

In [0]:
drivers_valid_df= (drivers_drop_df
                   .withColumnRenamed("driverId", "driver_id")
                   .withColumnRenamed("dateOfBirth", "date_of_birth"))

In [0]:
drivers_con_df= (drivers_valid_df
                 .withColumn("driver_name", F.initcap(F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName"))))
                 .drop ("name")
)

In [0]:
display(drivers_con_df)

In [0]:
drivers_dup_df= drivers_con_df.dropDuplicates(["driver_id"])

In [0]:
display(drivers_dup_df)

In [0]:
drivers_final_df= drivers_dup_df.withColumn("nationality",F.initcap("nationality"))

In [0]:
display(drivers_final_df)

In [0]:
( drivers_final_df
 .write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("silver_table"))

In [0]:
display(spark.table(f"{catalog_name}.{silver_schema}.drivers"))